#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

#Reading from bronze

In [0]:
df = spark.table("databricks_lakehouse.bronze.erp_loc_a101")
df.display()

#Transformation 

## Trimming

In [0]:
trim_plan = {
    field.name : F.trim(F.col(field.name))
    for field in df.schema.fields
    if isinstance(field.dataType, StringType)
}

df = df.withColumns(trim_plan)
df.display()

##CustID cleanup

In [0]:
df = df.withColumn("CID", F.regexp_replace(F.col("CID"),"-",""))
df.display()

##Normalizing country names

In [0]:
df = df.withColumn( "CNTRY",
    F.when(F.col("CNTRY")=="DE", "Germany")
    .when(F.col("CNTRY").isin("US","USA"), "United States")
    .when((F.col("CNTRY")=="") | (F.col("CNTRY").isNull()), "N/A")
    .otherwise(F.col("CNTRY"))
)
df.display()

##check for nulls

In [0]:
df.select([
    F.count(F.when(F.col(c).isNull(),c)).alias(c) for c in df.columns
]).display()

##Renameing columns

In [0]:
rename_map = {
    "CID": "customer_id",
    "CNTRY": "country"
}

df = df.withColumnsRenamed(rename_map)
df.display()

#Writing in silver

In [0]:
(
    df.write.mode("overwrite")
    .format("delta")
    .saveAsTable("databricks_lakehouse.silver.erp_customer_location")
)

#check silver table

In [0]:
%sql
select * from databricks_lakehouse.silver.erp_customer_location